# Instruct vs. Base Models — Fine-Tuning Llama 3.2 with LoRA

Large language models come in two common flavors: **base** models and **instruct** models. Knowing the difference is the key to fine-tuning them correctly.

| Aspect | Base model | Instruct model |
| --- | --- | --- |
| **Training** | Pretrained only — predicts the next token on raw text | Base model **+** instruction tuning (SFT / RLHF) |
| **Behaviour** | Continues / completes text; doesn't "follow" requests | Follows instructions, answers questions, holds a chat |
| **Input format** | Plain text | A **chat template** — `system` / `user` / `assistant` turns wrapped in special tokens |

An instruct model is trained to expect a **chat template** built from model-specific special tokens. For Llama 3.2 those tokens look like this:

```text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
...system prompt...<|eot_id|><|start_header_id|>user<|end_header_id|>
...user message...<|eot_id|><|start_header_id|>assistant<|end_header_id|>
```

A **base** model has never seen those tokens, so feeding it a chat template works poorly — and vice-versa. That single idea drives everything in this notebook.

---

## What this notebook does

Starting from the instruct model [`meta-llama/Llama-3.2-1B-Instruct`](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct), we fine-tune it to translate plain-English requests into **Docker commands**:

1. **Load** the model in 8-bit so it fits on a small GPU.
2. **Load a dataset** of natural-language → Docker-command pairs.
3. **Format** each example with Llama's chat template.
4. **Tokenize** and batch the data.
5. **Add LoRA adapters** and run supervised fine-tuning (SFT) — training only ~3.5% of the weights.
6. **Merge** the adapters into a full-precision model and push it to the Hub.
7. **Run inference** to watch the fine-tuned model in action.

> **Requirements:** a GPU runtime (e.g. Google Colab) and a Hugging Face token with access to Llama 3.2. The outputs below were captured from a real GPU run.

## 1. Setup & authentication

*Install the libraries and sign in to the Hugging Face Hub using a token from your `.env` file.*

In [ ]:
# --- Install dependencies ---
# transformers: models/tokenizers/Trainer   | datasets: data loading
# bitsandbytes: 8-bit quantization          | trl: SFTTrainer (SFT)
# peft: LoRA (parameter-efficient tuning)   | huggingface_hub: login + push
# python-dotenv: read the HF token from a local .env file
# (-q = quiet output, -U = upgrade to the latest versions)
!pip install -q -U transformers datasets bitsandbytes  trl peft  huggingface_hub python-dotenv

In [ ]:
# Authenticate with the Hugging Face Hub using a token from a local .env
# file (consistent with notebook 1). Required to download the gated Llama
# 3.2 model and to push your fine-tuned model to the Hub.
import os                          # read environment variables
from dotenv import load_dotenv     # load variables from a .env file
from huggingface_hub import login  # authenticate with the Hub

# Read the .env file in the current directory and set its keys as env vars.
load_dotenv()

# Fetch the token (os.getenv returns None if it isn't set).
hf_token = os.getenv("HF_TOKEN")

# Fail early with a clear message if the token is missing or still the placeholder.
if not hf_token or hf_token == "your_hugging_face_token_here":
    raise ValueError(
        "HF_TOKEN not found. Create a .env file with HF_TOKEN=hf_your_token_here "
        "(get a token at https://huggingface.co/settings/tokens)."
    )

# Log in for this session so downloads/uploads are authenticated.
login(token=hf_token)
print("Successfully authenticated with the Hugging Face Hub.")

Successfully authenticated with the Hugging Face Hub.


## 2. Load the instruct model (8-bit)

*Download Llama 3.2 1B Instruct and quantize it to 8-bit to save GPU memory.*

In [ ]:
# --- Load the INSTRUCT model in 8-bit ---
# We fine-tune the Instruct variant of Llama 3.2 1B (already chat-tuned).
# 8-bit quantization shrinks memory so it trains on a single modest GPU;
# device_map='auto' places the layers on the available GPU automatically.
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Alternative checkpoints you could swap in (kept for reference):
# model_name = "Cagatayd/llama3.2-1B-Instruct-Egitim"
# model_name = "unsloth/Llama-3.2-1B-Instruct"
model_name = "meta-llama/Llama-3.2-1B-Instruct"   # the instruct model we fine-tune

# Tell transformers to load the weights in 8-bit precision via bitsandbytes.
config_8bit = BitsAndBytesConfig(load_in_8bit=True)

# Download + instantiate the model with the 8-bit config.
model_8bit = AutoModelForCausalLM.from_pretrained(
    model_name,                        # which checkpoint to load
    quantization_config=config_8bit,   # load the weights in int8
    device_map="auto",                 # let accelerate place layers on the GPU
    trust_remote_code=True,            # allow the model's custom code if any
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


In [ ]:
# Inspect the architecture: the projections are Linear8bitLt (8-bit weights).
model_8bit   # printing a model shows its module tree

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

## 3. Load the tokenizer

*The tokenizer converts text to token ids and carries the model's chat template.*

In [ ]:
# Load the matching tokenizer. padding_side='left' matters for causal LMs at
# generation time so the most recent tokens stay at the right edge of a batch.
tokenizer = AutoTokenizer.from_pretrained(
    model_name,                # same checkpoint as the model
    padding_side="left",       # pad on the left (important for generation)
    trust_remote_code=True,    # allow custom tokenizer code if any
)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]


special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]


## 4. Load and split the dataset

*The dataset pairs an English request with the Docker command that satisfies it. We hold out 20% for validation.*

In [ ]:
# --- Load the training data ---
# dockerNLcommands: natural-language requests paired with the equivalent Docker
# CLI command. We teach the model this English -> command mapping.
from datasets import load_dataset

dataset = load_dataset("MattCoddity/dockerNLcommands")   # downloads + caches the dataset
dataset   # show the splits and their columns/rows

README.md:   0%|          | 0.00/1.46k [00:00<?, ?B/s]


06102023.json:   0%|          | 0.00/543k [00:00<?, ?B/s]


Generating train split:   0%|          | 0/2415 [00:00<?, ? examples/s]


DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [ ]:
# Peek at one raw example: an 'instruction', an 'input' (the NL request), and
# the target 'output' (the Docker command to produce).
dataset['train'][0]   # the first training row (a dict)

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [ ]:
# The dataset ships only a train split, so carve out 20% for validation.
from datasets import DatasetDict

# seed=42 makes the split reproducible; test_size=0.2 -> 80/20 train/val.
train_val_split = dataset['train'].train_test_split(test_size=0.2, seed=42)

# Re-assemble into a DatasetDict with clear 'train' and 'validation' keys.
dataset = DatasetDict(
    {
        'train': train_val_split['train'],
        'validation': train_val_split['test'],
    }
)
dataset   # now has both train and validation splits

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 483
    })
})

## 5. Format examples with a chat template

*Build `system` / `user` / `assistant` messages and render them with the model's chat template — the step that turns raw rows into instruct-style training text.*

In [ ]:
# --- Turn each row into a chat conversation ---
# Instruct models expect role-based messages. We map instruction->system,
# input->user, output->assistant. This 'messages' list is what the chat
# template will later render into the model's special-token format.
def to_chat_template(example):
    # Build the 3-turn conversation for this example.
    messages = [
        {"role": 'system', 'content': example['instruction']},   # task description
        {"role": 'user', 'content': example['input']},           # the user's request
        {"role": 'assistant', 'content': example['output']},     # the desired answer
    ]
    return {'text': messages}   # store the messages list under a new 'text' column

# .map applies the function to every row in every split.
dataset = dataset.map(to_chat_template)
dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]


Map:   0%|          | 0/483 [00:00<?, ? examples/s]


DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [ ]:
# Each 'text' field is now a list of role/content message dicts.
dataset['train']['text'][0]

[{'content': 'translate this sentence in docker command', 'role': 'system'},
 {'content': 'Find the repository, tag, and ID of the images that were created before the latest nginx image.',
  'role': 'user'},
 {'content': 'docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"',
  'role': 'assistant'}]

In [ ]:
# --- Why chat templates matter (Instruct vs Base) ---
# Different instruct models use different special tokens. Mistral's template
# wraps the conversation in [INST] ... [/INST] with <s>/</s> markers.
tokenizer_mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

# tokenize=False -> return the formatted STRING (not token ids) so we can read it.
tokenizer_mistral.apply_chat_template(dataset['train']['text'][0], tokenize=False)

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]


tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]


special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]


'<s> [INST] translate this sentence in docker command\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image. [/INST] docker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"</s>'

In [ ]:
# Llama 3.2's template instead uses <|begin_of_text|>, <|start_header_id|>,
# <|eot_id|>, etc. Same conversation, model-specific formatting -- exactly the
# structure a *base* (non-instruct) model was never trained to understand.
tokenizer.apply_chat_template(dataset['train']['text'][0], tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Feb 2025\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

In [ ]:
# Apply Llama's chat template to every example, replacing the messages list
# with the fully-formatted prompt string the model actually trains on.
def apply_chat_temp(example):
    # Render the messages list into Llama's special-token string.
    new_text = tokenizer.apply_chat_template(example['text'], tokenize=False)
    return {'text': new_text}   # overwrite 'text' with the templated string

dataset = dataset.map(apply_chat_temp)
dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]


Map:   0%|          | 0/483 [00:00<?, ? examples/s]


DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [ ]:
# The 'text' field is now the templated string with Llama's special tokens.
dataset['train']['text'][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Feb 2025\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the repository, tag, and ID of the images that were created before the latest nginx image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f "before=nginx:latest" --format "{{.Repository}},{{.Tag}},{{.ID}}"<|eot_id|>'

## 6. Tokenize the dataset

*Turn the formatted text into the `input_ids` the model trains on.*

In [ ]:
# --- Tokenize ---
# See how one templated example becomes input_ids + attention_mask.
tokenizer(dataset['train']['text'][0])   # returns a dict of input_ids + attention_mask

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1691, 13806, 220, 2366, 20, 271, 14372, 420, 11914, 304, 27686, 3290, 128009, 128006, 882, 128007, 271, 10086, 279, 12827, 11, 4877, 11, 323, 3110, 315, 279, 5448, 430, 1051, 3549, 1603, 279, 5652, 71582, 2217, 13, 128009, 128006, 78191, 128007, 271, 29748, 5448, 482, 69, 330, 15145, 28, 74661, 25, 19911, 1, 1198, 2293, 48319, 13, 4727, 39254, 3052, 13, 5786, 39254, 3052, 13, 926, 24275, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# Tokenize the whole dataset in batches and drop the raw text columns, leaving
# only input_ids / attention_mask for training.
def tokenize_fn(example):
    return tokenizer(example['text'])   # convert text -> token ids

# batched=True tokenizes many rows at once (faster); remove_columns drops the
# original text fields so only the model inputs remain.
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=['input', 'output', 'instruction', 'text'],
)
tokenized_dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]


Map:   0%|          | 0/483 [00:00<?, ? examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [ ]:
# Sequence lengths vary per example (here the 2nd training row is 79 tokens).
len(tokenized_dataset['train']['input_ids'][1])

79

## 7. Data collator & dynamic padding

*Batch variable-length sequences together and build the training labels automatically.*

In [ ]:
# --- Data collator ---
# DataCollatorForLanguageModeling(mlm=False) builds causal-LM batches: it pads
# sequences to equal length and creates the shifted 'labels' automatically.
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer,             # needed to know the pad token id
    mlm=False,             # causal LM (next-token), not masked LM
    return_tensors="pt",   # return PyTorch tensors
)

In [ ]:
# Llama has no dedicated pad token, so reuse the end-of-sequence token for padding.
tokenizer.pad_token = tokenizer.eos_token   # set <eos> as the pad token
tokenizer.pad_token                         # confirm it's set

'<|eot_id|>'

In [ ]:
# Sanity-check a few batches: each batch is padded to its own max length, so the
# sequence dimension changes from batch to batch (dynamic padding).
from torch.utils.data import DataLoader

# Build a loader that groups rows into batches of 2 using our collator.
data_loader = DataLoader(
    tokenized_dataset['train'],
    batch_size=2,
    collate_fn=data_collator,   # pads each batch + builds labels
)

# Iterate over the first few batches and print their tensor shapes.
for step, batch in enumerate(data_loader):
    print(f"Batch {step}")
    print("input_ids.shape:", batch["input_ids"].shape)            # (batch, seq_len)
    print("attention_mask.shape:", batch["attention_mask"].shape)  # 1 = real token, 0 = pad
    if step >= 3:   # stop after 4 batches
        break

Batch 0
input_ids.shape: torch.Size([2, 88])
attention_mask.shape: torch.Size([2, 88])
Batch 1
input_ids.shape: torch.Size([2, 79])
attention_mask.shape: torch.Size([2, 79])
Batch 2
input_ids.shape: torch.Size([2, 80])
attention_mask.shape: torch.Size([2, 80])
Batch 3
input_ids.shape: torch.Size([2, 78])
attention_mask.shape: torch.Size([2, 78])


In [ ]:
# Illustration of left-padding + label masking within a batch (batch size = 2).
# Shorter sequences are padded on the LEFT so the real tokens align to the right;
# padded positions get a label of -100 so the loss ignores them.
#
#   raw:     [1, 2, 3, 4, 5, 6]
#   padded:  [pad, pad, pad, 1, 2, 3]     labels at pad positions -> -100
#
#   raw:     [1, 2, 3, 4, 5]
#   padded:  [pad, pad, 1, 2, 3]          labels at pad positions -> -100


## Understanding LoRA (Low-Rank Adaptation)

Fine-tuning **every** weight of a 1.24-billion-parameter model is expensive: you must store gradients and optimizer state for *all* of them, and you end up with a full-size copy of the model for every task. **LoRA (Low-Rank Adaptation)** sidesteps this.

![LoRA: freeze the pretrained weights and learn a small low-rank update](images/lora-diagram.png)

**The idea.** For a pretrained weight matrix `W` (shape `d x d`), LoRA keeps `W` **frozen** and learns a small *low-rank* update instead of changing `W` directly:

$$\Delta W = B A, \qquad A \in \mathbb{R}^{r \times d},\; B \in \mathbb{R}^{d \times r},\; r \ll d$$

so the layer's output becomes:

$$h = W x + \frac{\alpha}{r}\, B A x$$

Only the small matrices `A` and `B` are trained; `W` never changes. Because the rank `r` is tiny compared to `d`, the number of trainable parameters collapses — in this notebook, from ~1.24B down to ~45M (**~3.5%**).

**Why we use LoRA here**

- **It fits on a small GPU.** Combined with 8-bit quantization, only the tiny adapters need gradients and optimizer state, so training runs on modest hardware.
- **It is faster and cheaper** to train than full fine-tuning.
- **The adapters are tiny and portable.** A saved adapter is a few MB (not gigabytes), so you can keep one base model and hot-swap task-specific adapters.
- **No catastrophic forgetting.** The base weights are untouched, so the model keeps its general abilities — and you can *merge* the adapter back into the weights later (we do exactly that near the end of this notebook).
- **Quality stays competitive** with full fine-tuning on many downstream tasks.

**The knobs you'll set in the next cell**

| Parameter | What it controls |
| --- | --- |
| `r` | Rank of the update — higher = more capacity and more parameters (here `64`) |
| `lora_alpha` | Scaling factor `alpha`; the update is scaled by `alpha / r` |
| `target_modules` | Which layers get adapters (the attention `q/k/v/o` and MLP projections) |
| `lora_dropout` | Dropout on the LoRA path, for regularization |
| `bias`, `task_type` | Whether to train biases; the task family (causal LM) |

## 8. Add LoRA adapters

*LoRA freezes the base weights and trains small low-rank matrices, so only a few percent of the parameters are updated.*

In [ ]:
# The base 8-bit model before adding LoRA (plain Linear8bitLt layers).
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

In [ ]:
# --- Add LoRA adapters (parameter-efficient fine-tuning) ---
# Full fine-tuning would update all ~1.24B weights. LoRA instead FREEZES the base
# model and injects small trainable rank-64 matrices (A and B) into each attention
# and MLP projection -- only those adapters are trained, saving memory and compute.
# The printout below confirms only ~3.5% of the parameters are trainable.
from peft import LoraConfig, get_peft_model
import copy

# Deep-copy so the original model_8bit stays adapter-free for comparison later.
model_8bit_clone = copy.deepcopy(model_8bit)

# Configure the LoRA adapters.
lora_config = LoraConfig(
    r=64,                    # rank of the low-rank update (more = more capacity)
    lora_alpha=32,           # scaling factor; the update is scaled by alpha/r
    target_modules=[         # which layers receive adapters:
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",      # MLP projections
    ],
    lora_dropout=0.05,       # dropout on the LoRA path (regularization)
    bias="none",             # do not train bias terms
    task_type="CAUSAL_LM",   # task family: causal language modeling
)

# Wrap the model so the forward pass adds the LoRA update; base weights stay frozen.
model_8bit_lora = get_peft_model(model_8bit_clone, lora_config)
model_8bit_lora.print_trainable_parameters()   # prints trainable vs total params

trainable params: 45,088,768 || all params: 1,280,903,168 || trainable%: 3.5201


In [ ]:
# The original 8-bit model is untouched (we deep-copied it before wrapping).
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMS

In [ ]:
# The LoRA-wrapped model: each target projection now has lora_A/lora_B adapters
# alongside the frozen base_layer.
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

## 9. Configure and run SFT training

*Set the training hyper-parameters and fine-tune with `trl`'s `SFTTrainer`.*

In [ ]:
# --- Training configuration ---
# Small demo run: 60 steps, batch size 2 x grad-accum 4 (effective 8), fp16,
# periodic eval/checkpoints. report_to='none' disables external loggers.
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./resutls",             # where checkpoints are written
    per_device_train_batch_size=2,      # training examples per GPU step
    per_device_eval_batch_size=2,       # eval examples per GPU step
    eval_steps=10,                      # evaluate every 10 steps...
    eval_strategy="steps",              # ...using a step-based schedule
    save_steps=20,                      # save a checkpoint every 20 steps
    save_strategy="steps",
    # num_train_epochs=1,               # (we use max_steps instead of epochs)
    max_steps=60,                       # stop after 60 optimizer steps
    learning_rate=3e-5,                 # optimizer step size
    weight_decay=0.01,                  # L2 regularization
    warmup_steps=5,                     # ramp the LR up over the first 5 steps
    fp16=True,                          # mixed-precision training (saves memory)
    gradient_accumulation_steps=4,      # accumulate 4 steps -> effective batch 8
    optim="adamw_torch",                # optimizer
    logging_steps=10,                   # log metrics every 10 steps
    report_to="none",                   # no W&B/TensorBoard logging
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Confirm the tokenized dataset the trainer will consume.
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [ ]:
# --- Train ---
# SFTTrainer (from trl) runs supervised fine-tuning: the model learns to predict the
# assistant's Docker command from the chat-formatted prompt. Watch the loss fall in
# the table below; the final TrainOutput reports the average loss over the 60 steps.
trainer = SFTTrainer(
    model=model_8bit_lora,                        # the LoRA-wrapped model to train
    train_dataset=tokenized_dataset['train'],     # training split
    eval_dataset=tokenized_dataset['validation'], # validation split
    args=training_args,                           # the config from the previous cell
    data_collator=data_collator,                  # pads batches + builds labels
    # tokenizer=tokenizer,                        # old argument name (deprecated)
    processing_class=tokenizer,                   # new argument name for the tokenizer
)

trainer.train()   # run the fine-tuning loop

<ipython-input-30-046adc83ed52>:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Converting train dataset to ChatML:   0%|          | 0/1932 [00:00<?, ? examples/s]


Applying chat template to train dataset:   0%|          | 0/1932 [00:00<?, ? examples/s]


Applying chat template to train dataset:   0%|          | 0/1932 [00:00<?, ? examples/s]


Converting eval dataset to ChatML:   0%|          | 0/483 [00:00<?, ? examples/s]


Applying chat template to eval dataset:   0%|          | 0/483 [00:00<?, ? examples/s]


Applying chat template to eval dataset:   0%|          | 0/483 [00:00<?, ? examples/s]


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss,Validation Loss
10,4.383600,3.290534
20,2.704600,2.147171
30,1.908200,1.652376
40,1.461900,1.330681
50,1.286300,1.208860
60,1.197200,1.184752


TrainOutput(global_step=60, training_loss=2.1569530646006267, metrics={'train_runtime': 252.0598, 'train_samples_per_second': 1.904, 'train_steps_per_second': 0.238, 'total_flos': 226414707916800.0, 'train_loss': 2.1569530646006267})

In [ ]:
# After training, the LoRA adapters hold the learned Docker-command behavior.
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

In [ ]:
# The frozen base weights are unchanged (still the original int8 values)...
import torch

# Print full numbers (no scientific notation) so the values are readable.
torch.set_printoptions(precision=8, sci_mode=False)

# Reach into the first decoder layer of the wrapped model.
layer0 = model_8bit_lora.model.model.layers[0]

# The base q_proj weight is still int8 and was NOT updated by training.
model_8bit_weights = layer0.self_attn.q_proj.weight
print("model_8bit_weights:", model_8bit_weights)

model_8bit_weights: Parameter containing:
Parameter(Int8Params([[-44,  16,  61,  ..., -22, -29,  50],
            [ 14,  70,  65,  ..., -39, -18,  13],
            [ 16,  14,  30,  ..., -34, -34, -24],
            ...,
            [ 17,  20,  40,  ..., -40, -15, -16],
            [ 32, -35,  50,  ..., -17, -41, -21],
            [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
           dtype=torch.int8))


In [ ]:
# ...while the LoRA A matrix is a small trainable fp32 tensor that WAS updated
# during training.
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

# LoRA A matrix (down-projection of the low-rank update); "default" is the adapter name.
weight_A = layer0.self_attn.q_proj.lora_A["default"].weight
print("LoRA A weight:", weight_A)

LoRA A weight: Parameter containing:
tensor([[ 0.01543376, -0.01957894,  0.01014431,  ...,  0.00173371,
          0.01613926, -0.01556054],
        [-0.01690101,  0.01074822,  0.01061925,  ...,  0.00292684,
         -0.00149638, -0.01090698],
        [-0.01426101,  0.01581102,  0.01960918,  ...,  0.00724101,
          0.01115141, -0.00878092],
        ...,
        [-0.01080369,  0.00528889, -0.01511446,  ..., -0.00140287,
          0.01316711, -0.01059285],
        [ 0.01593476,  0.01215826, -0.01826999,  ...,  0.01343037,
         -0.01595012, -0.00277690],
        [ 0.01049892, -0.00961566, -0.01900158,  ...,  0.00369666,
          0.00430331, -0.01020450]], device='cuda:0', requires_grad=True)


In [ ]:
# LoRA adapters train in float32 even though the base model is stored in int8.
weight_A.dtype

torch.float32

## 10. Merge the LoRA adapters into a full-precision model

*Adapters can't be merged into 8-bit weights, so we reload the model in fp32, attach the adapters, and fold them in.*

In [ ]:
# --- Prepare to merge: load the base model in full precision ---
# LoRA adapters cannot be merged into 8-bit weights, so we reload the model in
# default (fp32) precision to fold the adapters into it.
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,               # same checkpoint, but WITHOUT a quantization_config
    device_map="auto",        # place the layers on the GPU
    trust_remote_code=True,
)

In [ ]:
# Freshly loaded model: plain Linear layers, no adapters attached yet.
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [ ]:
# Save the trained LoRA adapters to disk so we can attach them to the fp32 model.
lora_path = "./lora_adapters"                 # output directory for the adapter files
model_8bit_lora.save_pretrained(lora_path)    # writes just the small adapter weights

In [ ]:
# Attach the saved LoRA adapters onto the full-precision base model.
from peft import PeftModel

base_model = PeftModel.from_pretrained(base_model, lora_path)   # wrap fp32 model with adapters
base_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
# Fold the adapters into the base weights and drop the LoRA wrappers, giving a
# standalone fine-tuned model (plain Linear layers again).
base_model = base_model.merge_and_unload()   # W <- W + (alpha/r) * B A, then remove adapters
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [ ]:
# Push the merged model + tokenizer to the Hub so they can be reused/shared.
# (Replace the repo id with your own namespace before running.)
base_model.push_to_hub("Cagatayd/Llama3.2-doker-egitim")   # upload the model weights
tokenizer.push_to_hub("Cagatayd/Llama3.2-doker-egitim")    # upload the tokenizer

model.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]


tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]


CommitInfo(commit_url='https://huggingface.co/Cagatayd/Llama3.2-doker-egitim/commit/883a2f201eb8efc3262705c890c88a6ba70ae3a0', commit_message='Upload tokenizer', commit_description='', oid='883a2f201eb8efc3262705c890c88a6ba70ae3a0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Cagatayd/Llama3.2-doker-egitim', endpoint='https://huggingface.co', repo_type='model', repo_id='Cagatayd/Llama3.2-doker-egitim'), pr_revision=None, pr_num=None)

## 11. Run inference with the fine-tuned model

*Give the model an English request and see the Docker command it generates.*

In [ ]:
# Optional: quick pipeline example against a pushed model id (left commented out).
# import torch
# from transformers import pipeline

# model_id = "Cagatayd/Llama3.2-doker-egitim"
# pipe = pipeline(
#     "text-generation",
#     model=model_id,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
# messages = [
#     {"role": "system", "content": "You are a pirate chatbot who always responds in pirate speak!"},
#     {"role": "user", "content": "Who are you?"},
# ]
# outputs = pipe(
#     messages,
#     max_new_tokens=256,
# )
# print(outputs[0]["generated_text"][-1])


In [ ]:
# --- Try the fine-tuned model ---
# A validation-style example: the NL request and its expected Docker command ('output').
dataset['train'][1]

{'input': 'Please show me the Docker containers that have exited and are related to the mongo image.',
 'output': "docker ps -a --filter 'status=exited' --filter 'ancestor=mongo'",
 'instruction': 'translate this sentence in docker command',
 'text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 21 Feb 2025\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nPlease show me the Docker containers that have exited and are related to the mongo image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker ps -a --filter 'status=exited' --filter 'ancestor=mongo'<|eot_id|>"}

In [ ]:
# Greedy generation (no sampling): feed a system+user chat prompt and print the
# model's generated Docker command. Compare it with the expected answer above to
# judge how well the fine-tuning worked.
import torch
from transformers import pipeline

# Build a text-generation pipeline around the merged model + tokenizer.
pipe = pipeline(
    "text-generation",
    model=base_model,
    torch_dtype=torch.bfloat16,   # run in bf16 for speed/memory
    device_map="auto",
    tokenizer=tokenizer,
)

# The chat prompt: a system instruction + the user's natural-language request.
messages = [
    {"role": "system", "content": "translate this sentence in docker command"},
    {"role": "user", "content": "Please show me the Docker containers that have exited and are related to the mongo image."},
]

# Generate up to 256 new tokens (greedy decoding by default).
outputs = pipe(
    messages,
    max_new_tokens=256,
)

# The generated assistant turn is the last message; print its text content.
print(outputs[0]["generated_text"][-1]['content'])

Device set to use cuda:0


docker ps --filter "status=exited" --filter "image= mongo"


In [ ]:
# Same prompt but with sampling (temperature/top_p/top_k) and a repetition
# penalty, which tends to give slightly cleaner, less repetitive output.
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    tokenizer=tokenizer,
)
messages = [
    {"role": "system", "content": "translate this sentence in docker command"},
    {"role": "user", "content": "Please show me the Docker containers that have exited and are related to the mongo image."},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
    temperature=0.5,           # lower = less random
    top_p=0.9,                 # nucleus sampling: keep the top 90% probability mass
    top_k=10,                  # only sample from the top 10 tokens
    do_sample=True,            # enable sampling (instead of greedy)
    repetition_penalty=1.2,    # discourage repeating the same tokens
)
print(outputs[0]["generated_text"][-1]['content'])

Device set to use cuda:0


docker ps --filter "status=exited" --filter "image=mongo"
